# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.11 — FAST
## Weak-Field Linearized Classical Benchmark Audit

### Mission

Le fond de référence hérité de `.3.3.10` est :

\[
h_{ij}=\delta_{ij},\qquad
N=1,\qquad
N^i=0,
\]

\[
s=1,\qquad v_i=0,\qquad
P=0,\qquad \lambda=0,
\]

sur une sous-branche ouverte de couplages où :

\[
\det Q_M\neq0,\qquad
\Delta_{{\rm FF},M}\neq0.
\]

On introduit :

\[
h_{ij}=\delta_{ij}+\varepsilon\,\gamma_{ij},
\qquad
N=1+\varepsilon\,n,
\qquad
N^i=\varepsilon\,n^i,
\]

\[
s=1+\varepsilon\,\sigma,
\qquad
v_i=\varepsilon\,w_i,
\qquad |\varepsilon|\ll1.
\]

### Discipline scientifique

Ce notebook ne déclarera `WEAK_FIELD_BENCHMARK_PASS=True` que si sont fermés :

1. les contraintes linéarisées ;
2. l'action quadratique pertinente ;
3. le comptage modal cohérent avec les 5 DOF canoniques ;
4. les secteurs tensoriel, vectoriel et scalaire ;
5. les conditions ghost/gradient ;
6. la limite statique faible champ.

Si seuls certains de ces éléments sont dérivés, le verdict restera `PARTIAL` et le notebook indiquera exactement ce qui manque.

In [1]:
# WF11.1 — Environment and frozen upstream provenance
from __future__ import annotations
import sympy as sp
import json, sys
from pathlib import Path

UPSTREAM = {
    "p3310": {
        "canonical_user_executed_sha256": "7fb04c381ce4671a884be97453ca45926c5a7f8fc67496c7f9b8ca3babf10c66",
        "canonical_user_executed_size_bytes": 33347,
        "MINKOWSKI_GVH_VACUUM_BENCHMARK_PASS_GENERIC_SUBBRANCH": True,
        "MINKOWSKI_GVH_VACUUM_BENCHMARK_PASS_ALL_COUPLINGS": False,
        "MINKOWSKI_BENCHMARK_STATUS": "PASS_ON_EXPLICIT_GENERIC_COUPLING_SUBBRANCH",
        "WEAK_FIELD_BENCHMARK_AUTHORIZED": True,
        "CLASSICAL_PREDICTIONS_AUTHORIZED": False,
        "QUANTIZATION_READY": False,
    },
    "p338": {
        "N_PHYSICAL_CONFIGURATION_DOF": 5,
        "FULL_GVH_DOF_COUNT_COMPUTED": True,
    },
}

UPSTREAM_GATE = all([
    UPSTREAM["p3310"]["MINKOWSKI_GVH_VACUUM_BENCHMARK_PASS_GENERIC_SUBBRANCH"],
    not UPSTREAM["p3310"]["MINKOWSKI_GVH_VACUUM_BENCHMARK_PASS_ALL_COUPLINGS"],
    UPSTREAM["p3310"]["WEAK_FIELD_BENCHMARK_AUTHORIZED"],
    not UPSTREAM["p3310"]["CLASSICAL_PREDICTIONS_AUTHORIZED"],
    not UPSTREAM["p3310"]["QUANTIZATION_READY"],
    UPSTREAM["p338"]["FULL_GVH_DOF_COUNT_COMPUTED"],
    UPSTREAM["p338"]["N_PHYSICAL_CONFIGURATION_DOF"] == 5,
])
assert UPSTREAM_GATE

print("Python =", sys.version.split()[0])
print("SymPy =", sp.__version__)
print("UPSTREAM_GATE =", UPSTREAM_GATE)
print("P3310_CANONICAL_SHA256 =", UPSTREAM["p3310"]["canonical_user_executed_sha256"])

Python = 3.12.13
SymPy = 1.14.0
UPSTREAM_GATE = True
P3310_CANONICAL_SHA256 = 7fb04c381ce4671a884be97453ca45926c5a7f8fc67496c7f9b8ca3babf10c66


# WF11.2 — Linéarisation de la contrainte de norme

La contrainte exacte est :

\[
\chi=-s^2+h^{ij}v_iv_j+1.
\]

À l'ordre \(\varepsilon\),

\[
s^2=1+2\varepsilon\sigma+\mathcal O(\varepsilon^2),
\]

tandis que :

\[
h^{ij}v_iv_j=\mathcal O(\varepsilon^2).
\]

Donc :

\[
\boxed{\chi^{(1)}=-2\sigma}.
\]

Sur la surface de contrainte :

\[
\boxed{\sigma=0}.
\]

Le mode normal \(s\) n'est donc pas un degré de liberté linéaire indépendant autour de ce fond.

In [2]:
# WF11.3 — Exact norm-constraint expansion
eps = sp.symbols("eps", real=True)
sigma = sp.symbols("sigma", real=True)
w1,w2,w3 = sp.symbols("w1 w2 w3", real=True)

s_eps = 1 + eps*sigma
# h^{-1} correction enters v^2 only at O(eps^3), so delta_ij suffices at O(eps^2).
chi_eps = sp.expand(-(s_eps**2) + eps**2*(w1**2+w2**2+w3**2) + 1)

chi_1 = sp.expand(chi_eps).coeff(eps,1)
chi_2 = sp.expand(chi_eps).coeff(eps,2)

LINEARIZED_NORM_CONSTRAINT_PASS = (chi_1 == -2*sigma)
SIGMA_LINEAR_CONSTRAINT = sp.Integer(0)

assert LINEARIZED_NORM_CONSTRAINT_PASS

print("chi^(1) =", chi_1)
print("chi^(2) =", chi_2)
print("LINEARIZED_NORM_CONSTRAINT_PASS =", LINEARIZED_NORM_CONSTRAINT_PASS)
print("sigma_linear =", SIGMA_LINEAR_CONSTRAINT)

chi^(1) = -2*sigma
chi^(2) = -sigma**2 + w1**2 + w2**2 + w3**2
LINEARIZED_NORM_CONSTRAINT_PASS = True
sigma_linear = 0


# WF11.4 — Blocs projetés au premier ordre

À partir des définitions amont :

\[
A=-\mathcal D_\perp s-v^ia_i^{(n)},
\]

\[
B_i=s\,a_i^{(n)}+\mathcal D_\perp v_i-K_i{}^jv_j,
\]

\[
C_i=-D_i s-K_i{}^jv_j,
\]

\[
D_{ij}=D_i v_j+sK_{ij},
\]

on obtient autour de Minkowski :

\[
A^{(1)}=-\dot\sigma,
\]

\[
B_i^{(1)}=\dot w_i+\partial_i n,
\]

\[
C_i^{(1)}=-\partial_i\sigma,
\]

\[
D_{ij}^{(1)}=\partial_iw_j+\kappa_{ij},
\]

où \(\kappa_{ij}=K_{ij}^{(1)}\).

Sur la surface \(\sigma=0\) :

\[
\boxed{A^{(1)}=C_i^{(1)}=0}.
\]

In [3]:
# WF11.5 — Symbolic linearized block ledger
sdot = sp.symbols("sdot", real=True)
n1,n2,n3 = sp.symbols("dn1 dn2 dn3", real=True)     # spatial gradient of lapse perturbation
wd1,wd2,wd3 = sp.symbols("wd1 wd2 wd3", real=True)  # time derivative of w_i
gs1,gs2,gs3 = sp.symbols("gs1 gs2 gs3", real=True)  # spatial gradient of sigma

k11,k22,k33,k12,k13,k23 = sp.symbols("k11 k22 k33 k12 k13 k23", real=True)
kappa = sp.Matrix([[k11,k12,k13],[k12,k22,k23],[k13,k23,k33]])
gradw = sp.Matrix(3,3, sp.symbols("gw0:9"))
gradn = sp.Matrix([n1,n2,n3])
wdot = sp.Matrix([wd1,wd2,wd3])
grads = sp.Matrix([gs1,gs2,gs3])

A1 = -sdot
B1 = wdot + gradn
C1 = -grads
D1 = gradw + kappa

# constrained linearized surface sigma=0 => its derivatives vanish
A1_c = sp.Integer(0)
C1_c = sp.zeros(3,1)

LINEARIZED_PROJECTED_BLOCKS_MATERIALIZED = all([
    A1 == -sdot,
    B1 == wdot + gradn,
    C1 == -grads,
    D1 == gradw + kappa,
])

assert LINEARIZED_PROJECTED_BLOCKS_MATERIALIZED

print("A1 =", A1)
print("B1 =", list(B1))
print("C1 =", list(C1))
print("LINEARIZED_PROJECTED_BLOCKS_MATERIALIZED =", LINEARIZED_PROJECTED_BLOCKS_MATERIALIZED)

A1 = -sdot
B1 = [dn1 + wd1, dn2 + wd2, dn3 + wd3]
C1 = [-gs1, -gs2, -gs3]
LINEARIZED_PROJECTED_BLOCKS_MATERIALIZED = True


# WF11.6 — Action quadratique du secteur GVH

Comme le fond vérifie :

\[
A=B=C=D=0,
\]

le premier terme non trivial du secteur vectoriel est quadratique.

Après \(\sigma=0\) :

\[
A^{(1)}=C_i^{(1)}=0.
\]

On a donc :

\[
I_1^{(2)}
=
-B_i^{(1)}B^{i(1)}
+
D_{ij}^{(1)}D^{ij(1)},
\]

\[
\theta^{(1)}=D_i{}^{i(1)},
\]

\[
I_3^{(2)}
=
D_{ij}^{(1)}D^{ji(1)},
\]

et :

\[
a^{2(2)}=B_i^{(1)}B^{i(1)}.
\]

Ainsi :

\[
\boxed{
\mathcal L_{u}^{(2)}
=
(c_1+c_4)B_iB^i
-c_1D_{ij}D^{ij}
-c_2(\mathrm{tr}D)^2
-c_3D_{ij}D^{ji}
}.
\]

In [4]:
# WF11.7 — Exact quadratic vector-sector action
c1,c2,c3,c4 = sp.symbols("c1 c2 c3 c4", real=True)

Bsq = sp.expand(B1.dot(B1))
DijDij = sp.expand(sum(D1[i,j]**2 for i in range(3) for j in range(3)))
trD = sp.expand(sp.trace(D1))
DijDji = sp.expand(sum(D1[i,j]*D1[j,i] for i in range(3) for j in range(3)))

Lu2 = sp.expand(
    (c1+c4)*Bsq
    - c1*DijDij
    - c2*trD**2
    - c3*DijDji
)

LINEARIZED_VECTOR_QUADRATIC_ACTION_DERIVED = True

# Independent reconstruction from invariant definitions on constrained surface.
I1_2 = sp.expand(-Bsq + DijDij)
theta_1 = trD
I3_2 = DijDji
a2_2 = Bsq
Lu2_reconstructed = sp.expand(-c1*I1_2 - c2*theta_1**2 - c3*I3_2 + c4*a2_2)

LINEARIZED_VECTOR_QUADRATIC_ACTION_CROSSCHECK_PASS = (
    sp.expand(Lu2 - Lu2_reconstructed) == 0
)

assert LINEARIZED_VECTOR_QUADRATIC_ACTION_CROSSCHECK_PASS

print("LINEARIZED_VECTOR_QUADRATIC_ACTION_DERIVED =", LINEARIZED_VECTOR_QUADRATIC_ACTION_DERIVED)
print("LINEARIZED_VECTOR_QUADRATIC_ACTION_CROSSCHECK_PASS =", LINEARIZED_VECTOR_QUADRATIC_ACTION_CROSSCHECK_PASS)

LINEARIZED_VECTOR_QUADRATIC_ACTION_DERIVED = True
LINEARIZED_VECTOR_QUADRATIC_ACTION_CROSSCHECK_PASS = True


# WF11.8 — Secteur tensoriel transverse-traceless

Pour isoler un premier sous-secteur physique sans imposer encore le secteur vectoriel/scalaires complet, on prend :

\[
\partial_i\gamma_{ij}^{TT}=0,
\qquad
\gamma_{ii}^{TT}=0,
\]

et :

\[
n=0,\qquad n^i=0,\qquad w_i=0.
\]

Alors :

\[
B_i^{(1)}=0,\qquad
D_{ij}^{(1)}=\kappa_{ij}^{TT}.
\]

Dans ce secteur :

\[
\mathrm{tr}\,D^{(1)}=0,
\qquad
D_{ij}^{(1)}D^{ji(1)}=D_{ij}^{(1)}D^{ij(1)}.
\]

Le secteur vectoriel modifie donc le coefficient cinétique tensoriel par :

\[
-(c_1+c_3)\kappa_{ij}\kappa^{ij}.
\]

Combiné au terme ADM \(K_{ij}K^{ij}-K^2\), le coefficient cinétique tensoriel devient :

\[
\boxed{1-c_1-c_3}.
\]

Ce notebook n'utilise pas ce résultat pour déclarer la stabilité complète : il classe seulement ce sous-secteur.

In [5]:
# WF11.9 — TT kinetic coefficient witness
kTT2 = sp.symbols("kTT2", nonnegative=True)
Lu2_TT = sp.expand(-(c1+c3)*kTT2)
LEH2_TT_kin = kTT2
L2_TT_kin = sp.expand(LEH2_TT_kin + Lu2_TT)

TENSOR_KINETIC_COEFFICIENT = sp.factor(sp.diff(L2_TT_kin, kTT2))
TENSOR_KINETIC_CLASSIFIED = (TENSOR_KINETIC_COEFFICIENT == 1-c1-c3)
TENSOR_NO_GHOST_CONDITION = sp.StrictGreaterThan(1-c1-c3, 0)

assert TENSOR_KINETIC_CLASSIFIED

print("TENSOR_KINETIC_COEFFICIENT =", TENSOR_KINETIC_COEFFICIENT)
print("TENSOR_NO_GHOST_CONDITION =", TENSOR_NO_GHOST_CONDITION)
print("TENSOR_KINETIC_CLASSIFIED =", TENSOR_KINETIC_CLASSIFIED)

TENSOR_KINETIC_COEFFICIENT = -c1 - c3 + 1
TENSOR_NO_GHOST_CONDITION = -c1 - c3 + 1 > 0
TENSOR_KINETIC_CLASSIFIED = True


# WF11.10 — Ce qui manque encore pour une dispersion tensorielle complète

Le coefficient cinétique tensoriel est maintenant dérivé.

Mais pour obtenir rigoureusement :

\[
\omega_T^2=c_T^2k^2,
\]

il faut fixer avec la même convention normalisée le terme spatial quadratique provenant de :

\[
{}^{(3)}R
\]

dans l'action EH complète.

Ce notebook ne remplace pas cette dérivation par une formule externe.

Par conséquent :

\[
\boxed{\texttt{TENSOR\_DISPERSION\_FULLY\_CLASSIFIED=False}}
\]

à ce stade, malgré le coefficient cinétique explicite.

In [6]:
# WF11.11 — Anti-overpromotion tensor gate
TENSOR_SPATIAL_GRADIENT_OPERATOR_DERIVED_IN_THIS_NOTEBOOK = False
TENSOR_DISPERSION_FULLY_CLASSIFIED = False
TENSOR_CAUSALITY_CLASSIFIED = False

TENSOR_SCOPE_DISCIPLINE_PASS = all([
    TENSOR_KINETIC_CLASSIFIED,
    not TENSOR_SPATIAL_GRADIENT_OPERATOR_DERIVED_IN_THIS_NOTEBOOK,
    not TENSOR_DISPERSION_FULLY_CLASSIFIED,
    not TENSOR_CAUSALITY_CLASSIFIED,
])

assert TENSOR_SCOPE_DISCIPLINE_PASS

print("TENSOR_SPATIAL_GRADIENT_OPERATOR_DERIVED_IN_THIS_NOTEBOOK =", TENSOR_SPATIAL_GRADIENT_OPERATOR_DERIVED_IN_THIS_NOTEBOOK)
print("TENSOR_DISPERSION_FULLY_CLASSIFIED =", TENSOR_DISPERSION_FULLY_CLASSIFIED)
print("TENSOR_CAUSALITY_CLASSIFIED =", TENSOR_CAUSALITY_CLASSIFIED)
print("TENSOR_SCOPE_DISCIPLINE_PASS =", TENSOR_SCOPE_DISCIPLINE_PASS)

TENSOR_SPATIAL_GRADIENT_OPERATOR_DERIVED_IN_THIS_NOTEBOOK = False
TENSOR_DISPERSION_FULLY_CLASSIFIED = False
TENSOR_CAUSALITY_CLASSIFIED = False
TENSOR_SCOPE_DISCIPLINE_PASS = True


# WF11.12 — Secteurs vectoriel et scalaire

Le lagrangien quadratique vectoriel complet est maintenant matérialisé en termes de :

\[
w_i,\quad n,\quad n^i,\quad \kappa_{ij},\quad \gamma_{ij}.
\]

Mais une vraie classification des modes exige encore :

1. décomposition scalaire-vectorielle-tensorielle (SVT) ;
2. élimination cohérente de \(n\) et \(n^i\) par leurs contraintes linéarisées ;
3. réduction de Dirac du quartet au niveau quadratique ;
4. diagonalisation des opérateurs cinétiques et gradients ;
5. vérification que le total propagatif correspond aux 5 DOF canoniques.

Ces étapes ne sont pas remplacées par un simple comptage de variables.

In [7]:
# WF11.13 — Mode-classification locks
SVT_DECOMPOSITION_MATERIALIZED = False
LINEARIZED_LAPSE_SHIFT_CONSTRAINTS_SOLVED = False
LINEARIZED_SECOND_CLASS_REDUCTION_MATERIALIZED = False
VECTOR_MODE_KINETICS_CLASSIFIED = False
SCALAR_MODE_KINETICS_CLASSIFIED = False
LINEARIZED_DOF_MATCH_PASS = False

MODE_CLASSIFICATION_PENDING = not all([
    SVT_DECOMPOSITION_MATERIALIZED,
    LINEARIZED_LAPSE_SHIFT_CONSTRAINTS_SOLVED,
    LINEARIZED_SECOND_CLASS_REDUCTION_MATERIALIZED,
    VECTOR_MODE_KINETICS_CLASSIFIED,
    SCALAR_MODE_KINETICS_CLASSIFIED,
    LINEARIZED_DOF_MATCH_PASS,
])

assert MODE_CLASSIFICATION_PENDING

print("SVT_DECOMPOSITION_MATERIALIZED =", SVT_DECOMPOSITION_MATERIALIZED)
print("LINEARIZED_LAPSE_SHIFT_CONSTRAINTS_SOLVED =", LINEARIZED_LAPSE_SHIFT_CONSTRAINTS_SOLVED)
print("VECTOR_MODE_KINETICS_CLASSIFIED =", VECTOR_MODE_KINETICS_CLASSIFIED)
print("SCALAR_MODE_KINETICS_CLASSIFIED =", SCALAR_MODE_KINETICS_CLASSIFIED)
print("LINEARIZED_DOF_MATCH_PASS =", LINEARIZED_DOF_MATCH_PASS)
print("MODE_CLASSIFICATION_PENDING =", MODE_CLASSIFICATION_PENDING)

SVT_DECOMPOSITION_MATERIALIZED = False
LINEARIZED_LAPSE_SHIFT_CONSTRAINTS_SOLVED = False
VECTOR_MODE_KINETICS_CLASSIFIED = False
SCALAR_MODE_KINETICS_CLASSIFIED = False
LINEARIZED_DOF_MATCH_PASS = False
MODE_CLASSIFICATION_PENDING = True


# WF11.14 — Limite statique faible champ

Le benchmark faible champ complet doit aussi déterminer si le secteur statique fournit une équation de type Poisson :

\[
\nabla^2\Phi
=
4\pi G_{\rm eff}\rho,
\]

et calculer \(G_{\rm eff}\) dans les conventions GVH.

Aucune valeur de \(G_{\rm eff}\) n'est supposée ici.

Il faut dériver le secteur scalaire statique de l'action quadratique couplée, résoudre les contraintes de lapse/shift/champ vectoriel et comparer le potentiel obtenu au régime newtonien.

Donc :

\[
\boxed{\texttt{STATIC\_WEAK\_FIELD\_LIMIT\_CLASSIFIED=False}}.
\]

In [8]:
# WF11.15 — Static weak-field lock
STATIC_WEAK_FIELD_LIMIT_CLASSIFIED = False
POISSON_EQUATION_DERIVED = False
G_EFFECTIVE_DERIVED = False
PPN_READY = False

STATIC_SCOPE_DISCIPLINE_PASS = all([
    not STATIC_WEAK_FIELD_LIMIT_CLASSIFIED,
    not POISSON_EQUATION_DERIVED,
    not G_EFFECTIVE_DERIVED,
    not PPN_READY,
])

assert STATIC_SCOPE_DISCIPLINE_PASS

print("STATIC_WEAK_FIELD_LIMIT_CLASSIFIED =", STATIC_WEAK_FIELD_LIMIT_CLASSIFIED)
print("POISSON_EQUATION_DERIVED =", POISSON_EQUATION_DERIVED)
print("G_EFFECTIVE_DERIVED =", G_EFFECTIVE_DERIVED)
print("PPN_READY =", PPN_READY)

STATIC_WEAK_FIELD_LIMIT_CLASSIFIED = False
POISSON_EQUATION_DERIVED = False
G_EFFECTIVE_DERIVED = False
PPN_READY = False


# WF11.16 — Verdict scientifique

`.3.3.11 FAST` établit déjà :

- la contrainte linéarisée \(\sigma=0\) ;
- les blocs projetés au premier ordre ;
- l'action quadratique du secteur GVH ;
- le coefficient cinétique tensoriel \(1-c_1-c_3\).

Mais il ne doit pas déclarer le benchmark faible champ complet tant que ne sont pas fermés :

- le gradient tensoriel normalisé ;
- les secteurs vectoriel et scalaire ;
- le matching des 5 DOF ;
- la limite statique de Poisson.

Le verdict correct est donc attendu comme :

\[
\boxed{\texttt{PARTIAL\_PASS}}
\]

avec un prochain maillon ciblé sur la réduction SVT et les contraintes linéarisées.

In [9]:
# WF11.17 — Final scientific classifier
LINEARIZED_CONSTRAINTS_PARTIAL_PASS = LINEARIZED_NORM_CONSTRAINT_PASS
QUADRATIC_VECTOR_ACTION_DERIVED = all([
    LINEARIZED_PROJECTED_BLOCKS_MATERIALIZED,
    LINEARIZED_VECTOR_QUADRATIC_ACTION_DERIVED,
    LINEARIZED_VECTOR_QUADRATIC_ACTION_CROSSCHECK_PASS,
])

WEAK_FIELD_CORE_LINEARIZATION_PASS = all([
    UPSTREAM_GATE,
    LINEARIZED_CONSTRAINTS_PARTIAL_PASS,
    QUADRATIC_VECTOR_ACTION_DERIVED,
    TENSOR_KINETIC_CLASSIFIED,
    TENSOR_SCOPE_DISCIPLINE_PASS,
    MODE_CLASSIFICATION_PENDING,
    STATIC_SCOPE_DISCIPLINE_PASS,
])

WEAK_FIELD_BENCHMARK_PASS = all([
    WEAK_FIELD_CORE_LINEARIZATION_PASS,
    TENSOR_DISPERSION_FULLY_CLASSIFIED,
    LINEARIZED_DOF_MATCH_PASS,
    VECTOR_MODE_KINETICS_CLASSIFIED,
    SCALAR_MODE_KINETICS_CLASSIFIED,
    STATIC_WEAK_FIELD_LIMIT_CLASSIFIED,
])

WEAK_FIELD_BENCHMARK_STATUS = (
    "PASS"
    if WEAK_FIELD_BENCHMARK_PASS
    else "PARTIAL_PASS_LINEARIZATION_AND_TENSOR_KINETIC_ONLY"
)

SCHWARZSCHILD_BENCHMARK_AUTHORIZED = WEAK_FIELD_BENCHMARK_PASS
CLASSICAL_PREDICTIONS_AUTHORIZED = False
QUANTIZATION_READY = False

WF11_OBSTRUCTIONS = []
if not TENSOR_DISPERSION_FULLY_CLASSIFIED:
    WF11_OBSTRUCTIONS.append("DERIVE-NORMALIZED-TENSOR-GRADIENT-AND-DISPERSION")
if not SVT_DECOMPOSITION_MATERIALIZED:
    WF11_OBSTRUCTIONS.append("MATERIALIZE-SVT-DECOMPOSITION")
if not LINEARIZED_LAPSE_SHIFT_CONSTRAINTS_SOLVED:
    WF11_OBSTRUCTIONS.append("SOLVE-LINEARIZED-LAPSE-SHIFT-CONSTRAINTS")
if not LINEARIZED_SECOND_CLASS_REDUCTION_MATERIALIZED:
    WF11_OBSTRUCTIONS.append("MATERIALIZE-LINEARIZED-SECOND-CLASS-REDUCTION")
if not LINEARIZED_DOF_MATCH_PASS:
    WF11_OBSTRUCTIONS.append("MATCH-LINEARIZED-MODES-TO-5-CANONICAL-DOF")
if not STATIC_WEAK_FIELD_LIMIT_CLASSIFIED:
    WF11_OBSTRUCTIONS.append("DERIVE-STATIC-POISSON-LIMIT-AND-G_EFFECTIVE")

WF11_LOCAL_AUDIT_PASS = all([
    WEAK_FIELD_CORE_LINEARIZATION_PASS,
    not WEAK_FIELD_BENCHMARK_PASS,
    WEAK_FIELD_BENCHMARK_STATUS == "PARTIAL_PASS_LINEARIZATION_AND_TENSOR_KINETIC_ONLY",
    not SCHWARZSCHILD_BENCHMARK_AUTHORIZED,
    not CLASSICAL_PREDICTIONS_AUTHORIZED,
    not QUANTIZATION_READY,
    len(WF11_OBSTRUCTIONS) > 0,
])

WF11_NEXT_AUTHORIZED = (
    "AUDIT-LINEARIZED-SVT-CONSTRAINT-REDUCTION-AND-TENSOR-DISPERSION"
    if WF11_LOCAL_AUDIT_PASS
    else "REPAIR-.3.3.11-LINEARIZED-CORE"
)

assert WF11_LOCAL_AUDIT_PASS

print("LINEARIZED_NORM_CONSTRAINT_PASS =", LINEARIZED_NORM_CONSTRAINT_PASS)
print("LINEARIZED_PROJECTED_BLOCKS_MATERIALIZED =", LINEARIZED_PROJECTED_BLOCKS_MATERIALIZED)
print("LINEARIZED_VECTOR_QUADRATIC_ACTION_DERIVED =", LINEARIZED_VECTOR_QUADRATIC_ACTION_DERIVED)
print("TENSOR_KINETIC_COEFFICIENT =", TENSOR_KINETIC_COEFFICIENT)
print("TENSOR_KINETIC_CLASSIFIED =", TENSOR_KINETIC_CLASSIFIED)
print("WEAK_FIELD_CORE_LINEARIZATION_PASS =", WEAK_FIELD_CORE_LINEARIZATION_PASS)
print("WEAK_FIELD_BENCHMARK_PASS =", WEAK_FIELD_BENCHMARK_PASS)
print("WEAK_FIELD_BENCHMARK_STATUS =", WEAK_FIELD_BENCHMARK_STATUS)
print("SCHWARZSCHILD_BENCHMARK_AUTHORIZED =", SCHWARZSCHILD_BENCHMARK_AUTHORIZED)
print("WF11_OBSTRUCTIONS =", WF11_OBSTRUCTIONS)
print("WF11_NEXT_AUTHORIZED =", WF11_NEXT_AUTHORIZED)

LINEARIZED_NORM_CONSTRAINT_PASS = True
LINEARIZED_PROJECTED_BLOCKS_MATERIALIZED = True
LINEARIZED_VECTOR_QUADRATIC_ACTION_DERIVED = True
TENSOR_KINETIC_COEFFICIENT = -c1 - c3 + 1
TENSOR_KINETIC_CLASSIFIED = True
WEAK_FIELD_CORE_LINEARIZATION_PASS = True
WEAK_FIELD_BENCHMARK_PASS = False
WEAK_FIELD_BENCHMARK_STATUS = PARTIAL_PASS_LINEARIZATION_AND_TENSOR_KINETIC_ONLY
SCHWARZSCHILD_BENCHMARK_AUTHORIZED = False
WF11_OBSTRUCTIONS = ['DERIVE-NORMALIZED-TENSOR-GRADIENT-AND-DISPERSION', 'MATERIALIZE-SVT-DECOMPOSITION', 'SOLVE-LINEARIZED-LAPSE-SHIFT-CONSTRAINTS', 'MATERIALIZE-LINEARIZED-SECOND-CLASS-REDUCTION', 'MATCH-LINEARIZED-MODES-TO-5-CANONICAL-DOF', 'DERIVE-STATIC-POISSON-LIMIT-AND-G_EFFECTIVE']
WF11_NEXT_AUTHORIZED = AUDIT-LINEARIZED-SVT-CONSTRAINT-REDUCTION-AND-TENSOR-DISPERSION


# WF11.18 — Portée du résultat

Un PASS local de ce notebook signifie seulement que la **linéarisation de base** est matériellement construite.

Le résultat physique nouveau le plus concret est :

\[
\boxed{
K_T(c_A)=1-c_1-c_3
}
\]

pour le sous-secteur tensoriel TT dans les conventions de ce notebook.

La condition nécessaire d'absence de ghost tensoriel est donc :

\[
\boxed{
1-c_1-c_3>0
}
\]

mais cela ne suffit pas encore à conclure à la stabilité complète, car le terme gradient et les autres secteurs doivent être classifiés.

Aucune autorisation Schwarzschild n'est donnée à ce stade.

In [10]:
# WF11.19 — Machine-readable artifact
artifact = {
    "notebook": "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.11_Weak_Field_Linearized_Classical_Benchmark_Audit_FAST",
    "execution_scope": "LINEARIZED_AROUND_MINKOWSKI_GENERIC_COUPLING_SUBBRANCH_BOR",
    "upstream": UPSTREAM,
    "linearized_constraint": {
        "chi_1": str(chi_1),
        "sigma_linear": 0,
        "pass": LINEARIZED_NORM_CONSTRAINT_PASS,
    },
    "linearized_blocks": {
        "A1": str(A1),
        "B1": [str(x) for x in B1],
        "C1": [str(x) for x in C1],
        "D1": [[str(D1[i,j]) for j in range(3)] for i in range(3)],
        "materialized": LINEARIZED_PROJECTED_BLOCKS_MATERIALIZED,
    },
    "quadratic_vector_sector": {
        "formula": "(c1+c4) B_i B_i - c1 D_ij D_ij - c2 (tr D)^2 - c3 D_ij D_ji",
        "derived": LINEARIZED_VECTOR_QUADRATIC_ACTION_DERIVED,
        "crosscheck_pass": LINEARIZED_VECTOR_QUADRATIC_ACTION_CROSSCHECK_PASS,
    },
    "tensor_sector": {
        "kinetic_coefficient": str(TENSOR_KINETIC_COEFFICIENT),
        "no_ghost_condition": str(TENSOR_NO_GHOST_CONDITION),
        "kinetic_classified": TENSOR_KINETIC_CLASSIFIED,
        "spatial_gradient_derived": TENSOR_SPATIAL_GRADIENT_OPERATOR_DERIVED_IN_THIS_NOTEBOOK,
        "dispersion_fully_classified": TENSOR_DISPERSION_FULLY_CLASSIFIED,
        "causality_classified": TENSOR_CAUSALITY_CLASSIFIED,
    },
    "pending": {
        "SVT_DECOMPOSITION_MATERIALIZED": SVT_DECOMPOSITION_MATERIALIZED,
        "LINEARIZED_LAPSE_SHIFT_CONSTRAINTS_SOLVED": LINEARIZED_LAPSE_SHIFT_CONSTRAINTS_SOLVED,
        "LINEARIZED_SECOND_CLASS_REDUCTION_MATERIALIZED": LINEARIZED_SECOND_CLASS_REDUCTION_MATERIALIZED,
        "VECTOR_MODE_KINETICS_CLASSIFIED": VECTOR_MODE_KINETICS_CLASSIFIED,
        "SCALAR_MODE_KINETICS_CLASSIFIED": SCALAR_MODE_KINETICS_CLASSIFIED,
        "LINEARIZED_DOF_MATCH_PASS": LINEARIZED_DOF_MATCH_PASS,
        "STATIC_WEAK_FIELD_LIMIT_CLASSIFIED": STATIC_WEAK_FIELD_LIMIT_CLASSIFIED,
        "POISSON_EQUATION_DERIVED": POISSON_EQUATION_DERIVED,
        "G_EFFECTIVE_DERIVED": G_EFFECTIVE_DERIVED,
    },
    "scientific_status": {
        "WEAK_FIELD_CORE_LINEARIZATION_PASS": WEAK_FIELD_CORE_LINEARIZATION_PASS,
        "WEAK_FIELD_BENCHMARK_PASS": WEAK_FIELD_BENCHMARK_PASS,
        "WEAK_FIELD_BENCHMARK_STATUS": WEAK_FIELD_BENCHMARK_STATUS,
        "SCHWARZSCHILD_BENCHMARK_AUTHORIZED": SCHWARZSCHILD_BENCHMARK_AUTHORIZED,
        "CLASSICAL_PREDICTIONS_AUTHORIZED": CLASSICAL_PREDICTIONS_AUTHORIZED,
        "QUANTIZATION_READY": QUANTIZATION_READY,
    },
    "verdict": {
        "WF11_LOCAL_AUDIT_PASS": WF11_LOCAL_AUDIT_PASS,
        "obstructions": WF11_OBSTRUCTIONS,
    },
    "next_authorized": WF11_NEXT_AUTHORIZED,
    "scope_note": "Partial weak-field audit: linearized norm constraint, projected blocks, quadratic vector sector and TT kinetic coefficient derived; full SVT reduction, dispersion and Poisson limit remain open."
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True, exist_ok=True)
artifact_path = export_dir / "gvh_0.3.2.7.3.7.3.3.11_Weak_Field_Linearized_Classical_Benchmark_Audit_FAST.json"
artifact_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")
print("WF11 artifact =", artifact_path)

WF11 artifact = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.11_Weak_Field_Linearized_Classical_Benchmark_Audit_FAST.json


# Conclusion

`.3.3.11 FAST` construit la base linéarisée sans sur-promouvoir le résultat.

Il établit :

\[
\boxed{\sigma=0}
\]

au premier ordre,

\[
\boxed{
\mathcal L_u^{(2)}
=
(c_1+c_4)B^2
-c_1D_{ij}D^{ij}
-c_2(\mathrm{tr}D)^2
-c_3D_{ij}D^{ji}
}
\]

et :

\[
\boxed{
K_T=1-c_1-c_3
}
\]

dans le secteur TT.

Mais le benchmark faible champ complet reste ouvert tant que la réduction SVT, la dispersion complète et la limite de Poisson ne sont pas dérivées.

Le prochain maillon autorisé est :

\[
\boxed{
\texttt{AUDIT-LINEARIZED-SVT-CONSTRAINT-REDUCTION-AND-TENSOR-DISPERSION}.
}
\]